# 🏨 Hotel Cancellation Predictor — Google Colab Quickstart

Run the whole project in Colab: clone → install → train → predict, plus an optional public link to the Streamlit app.

> Runtime tip: **Runtime → Run all**. No GPU needed.

## 1. Clone the repository and install the package

In [ ]:
REPO_URL = "https://github.com/theresamariaMagdalena/prediksi-pembatalan-kamar.git"

import os

if not os.path.exists("prediksi-pembatalan-kamar"):
    !git clone $REPO_URL
%cd prediksi-pembatalan-kamar
!pip install -q -e .

## 2. Confirm the dataset is present

In [ ]:
import pandas as pd
from hotel_cancel.config import Config
from hotel_cancel.data import load_dataset

config = Config.load()
X, y = load_dataset(config.raw_data_path)
print("rows:", len(X), "| features:", X.shape[1])
print("cancellation rate:", round(y.mean(), 3))
X.head()

## 3. Train the model

Writes `models/pipeline.pkl` and `models/metadata.pkl`, and saves diagnostic figures under `reports/figures/`.

In [ ]:
!python -m hotel_cancel.train

## 4. Inspect the diagnostic figures

In [ ]:
from IPython.display import Image, display

for name in ["confusion_matrix.png", "roc_curve.png", "feature_importances.png"]:
    path = config.figures_dir / name
    if path.exists():
        display(Image(filename=str(path)))

## 5. Score a single booking (mimics the app)

In [ ]:
from hotel_cancel.predict import load_artifacts, predict_proba

pipeline, metadata = load_artifacts(config)

booking = {
    "lead_time": 120, "adults": 2, "children": 1, "babies": 0,
    "stays_in_weekend_nights": 2, "stays_in_week_nights": 3,
    "previous_cancellations": 1, "booking_changes": 0, "adr": 95.5,
    "required_car_parking_spaces": 0, "total_of_special_requests": 1,
    "hotel": "City Hotel", "meal": "BB",
    "market_segment": "Online TA", "deposit_type": "No Deposit",
}
proba = predict_proba(booking, pipeline, metadata)
print(f"Cancellation probability: {proba:.1%}")

## 6. (Optional) Run the Streamlit app from Colab

Colab can't open `localhost`, so we expose the app through a public tunnel. Run the cell, then click the printed URL (enter the IP shown by the tunnel as the password if asked).

In [ ]:
!pip install -q streamlit
!npm install -g localtunnel 2>/dev/null
!curl -s https://loca.lt/mytunnelpassword  # this IP is the tunnel password
!streamlit run app/streamlit_app.py &>/content/streamlit.log &
import time; time.sleep(6)
!npx localtunnel --port 8501